# Phase 4: Integration 3: Cross-Perspective Synthesis

## Overview
Generic integration analyses that combine all three perspectives (structural, functional,
representational) to produce unified summaries. No task-specific hypothesis testing;
just systematic comparison of what each perspective says and where they agree or disagree.

## Analyses
1. **Variance decomposition**: How much variance in each metric is explained by model vs condition vs replication?
2. **Cross-perspective concordance**: Do metrics from different perspectives agree on circuit rankings?
3. **Condition effect consistency**: Do conditions rank the same across perspectives?
4. **Scaling consistency**: Do all perspectives show the same scaling trends with model size?
5. **Perspective disagreement detection**: Which circuits are most paradoxical (perspectives disagree)?

## Output Files
- `unified_variance_decomposition.csv`, `cross_perspective_concordance.csv`
- `condition_ranking_consistency.csv`, `scaling_consistency.csv`
- `perspective_disagreements.csv`
- Visualizations: T4_05 through T4_08

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
PROJECT_ROOT = Path("LSC_circuit_analysis")
PHASE4_DIR = PROJECT_ROOT / "04_Phase_Integration"
ANALYSIS_DIR = PHASE4_DIR / "outputs" / "analysis"
VIZ_DIR = PHASE4_DIR / "outputs" / "viz"

# Load unified tables from NB01
df_pc = pd.read_csv(ANALYSIS_DIR / "unified_per_circuit.csv")
df_pw = pd.read_csv(ANALYSIS_DIR / "unified_pairwise.csv")
df_ml = pd.read_csv(ANALYSIS_DIR / "unified_model_level.csv")
df_cat = pd.read_csv(ANALYSIS_DIR / "metric_catalog.csv")

# Load triangle edge results from NB02
df_sf = pd.read_csv(ANALYSIS_DIR / "triangle_sf_edge.csv")
df_sr = pd.read_csv(ANALYSIS_DIR / "triangle_sr_edge.csv")
df_rf = pd.read_csv(ANALYSIS_DIR / "triangle_rf_edge.csv")
df_pred = pd.read_csv(ANALYSIS_DIR / "incremental_prediction.csv")

# Metric groups
perspective_map = dict(zip(df_cat["metric"], df_cat["perspective"]))
S_METRICS = df_cat[df_cat["perspective"] == "structural"]["metric"].tolist()
F_METRICS = df_cat[df_cat["perspective"] == "functional"]["metric"].tolist()
R_METRICS = df_cat[df_cat["perspective"] == "representational"]["metric"].tolist()

# Filter to metrics in per-circuit table with variance
ALL_METRICS = [
    m
    for m in S_METRICS + F_METRICS + R_METRICS
    if m in df_pc.columns and df_pc[m].std() > 1e-10
]

MODELS = sorted(df_pc["model"].unique())
CONDITIONS = sorted(df_pc["band"].unique())

MODEL_COLORS = {
    "pythia-70m": "#1f77b4",
    "pythia-160m": "#ff7f0e",
    "pythia-410m": "#2ca02c",
    "pythia-1b": "#d62728",
    "pythia-1.4b": "#9467bd",
}
PERSPECTIVE_COLORS = {
    "structural": "#1f77b4",
    "functional": "#ff7f0e",
    "representational": "#2ca02c",
}

print(f"Per-circuit: {df_pc.shape}")
print(f"All metrics with variance: {len(ALL_METRICS)}")
print(f"Models: {MODELS}")
print(f"Conditions: {CONDITIONS}")

Per-circuit: (60, 32)
All metrics with variance: 26
Models: ['pythia-1.4b', 'pythia-160m', 'pythia-1b', 'pythia-410m', 'pythia-70m']
Conditions: ['high', 'low', 'medium', 'very_high']


## 1. Unified Variance Decomposition

For each metric, compute eta² (fraction of variance explained) by model, condition, and replication.
Group results by perspective to see if the dominance pattern is consistent.

In [2]:
def compute_eta_squared(df, metric, factor):
    """Compute eta² for a single metric and grouping factor."""
    groups = [g[metric].values for _, g in df.groupby(factor)]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return np.nan
    grand_mean = df[metric].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((df[metric] - grand_mean) ** 2).sum()
    return ss_between / ss_total if ss_total > 0 else 0.0


var_rows = []
for metric in ALL_METRICS:
    eta2_model = compute_eta_squared(df_pc, metric, "model")
    eta2_cond = compute_eta_squared(df_pc, metric, "band")
    eta2_repl = compute_eta_squared(df_pc, metric, "draw")
    residual = max(0, 1 - eta2_model - eta2_cond - eta2_repl)

    var_rows.append(
        {
            "metric": metric,
            "perspective": perspective_map.get(metric, "unknown"),
            "eta2_model": eta2_model,
            "eta2_condition": eta2_cond,
            "eta2_replication": eta2_repl,
            "residual": residual,
            "dominant_factor": ["model", "condition", "replication"][
                np.argmax([eta2_model, eta2_cond, eta2_repl])
            ],
        }
    )

df_var = pd.DataFrame(var_rows)
df_var.to_csv(ANALYSIS_DIR / "unified_variance_decomposition.csv", index=False)

print(f"Variance decomposition for {len(df_var)} metrics:")
print(
    df_var[
        [
            "metric",
            "perspective",
            "eta2_model",
            "eta2_condition",
            "eta2_replication",
            "dominant_factor",
        ]
    ].to_string(index=False)
)

print(f"\nMean eta² by perspective:")
print(
    df_var.groupby("perspective")[["eta2_model", "eta2_condition", "eta2_replication"]]
    .mean()
    .to_string()
)

Variance decomposition for 26 metrics:
                 metric      perspective  eta2_model  eta2_condition  eta2_replication dominant_factor
          edge_fraction       structural    0.995733    2.161103e-04          0.000023           model
          skip_fraction       structural    0.992135    2.445612e-03          0.000085           model
          attn_fraction       structural    0.778857    2.635505e-02          0.009142           model
           mlp_fraction       structural    0.840215    1.089353e-02          0.006208           model
         resid_fraction       structural    0.956581    1.569933e-02          0.001465           model
head_participation_rate       structural    0.963915    5.731882e-03          0.000078           model
           active_heads       structural    0.987903    2.951524e-03          0.000073           model
    mean_edges_per_head       structural    0.861778    4.530728e-02          0.004655           model
            n_universal       stru

In [3]:
# --- VIZ T4_05: Variance Decomposition by Perspective ---

fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Sort metrics by perspective then by eta2_model
df_plot = df_var.sort_values(
    ["perspective", "eta2_model"], ascending=[True, False]
).reset_index(drop=True)

x = np.arange(len(df_plot))
bar_width = 0.8

# Stacked bars
bottom = np.zeros(len(df_plot))
for factor, color, label in [
    ("eta2_model", "#1f77b4", "Model"),
    ("eta2_condition", "#ff7f0e", "Condition"),
    ("eta2_replication", "#2ca02c", "Replication"),
    ("residual", "#cccccc", "Residual"),
]:
    vals = df_plot[factor].values
    ax.bar(x, vals, bar_width, bottom=bottom, color=color, label=label, alpha=0.85)
    bottom += vals

# Mark perspective boundaries
perspectives = df_plot["perspective"].values
prev = perspectives[0]
for i in range(1, len(perspectives)):
    if perspectives[i] != prev:
        ax.axvline(x=i - 0.5, color="black", linestyle="--", alpha=0.3)
        # Label perspective
        start = np.where(perspectives == prev)[0][0]
        mid = (start + i - 1) / 2
        ax.text(
            mid,
            1.05,
            prev.capitalize(),
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )
        prev = perspectives[i]
# Last perspective
start = np.where(perspectives == prev)[0][0]
mid = (start + len(perspectives) - 1) / 2
ax.text(
    mid,
    1.05,
    prev.capitalize(),
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
)

ax.set_xticks(x)
ax.set_xticklabels(df_plot["metric"].values, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Fraction of Variance (eta²)", fontsize=11)
ax.set_title(
    "Variance Decomposition by Metric and Perspective", fontsize=13, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig(VIZ_DIR / "T4_05_variance_decomposition.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_05_variance_decomposition.png")

Saved: T4_05_variance_decomposition.png


## 2. Cross-Perspective Concordance

Compute Spearman rank correlation between all pairs of metrics across perspectives.
Build a concordance matrix and cluster it to reveal blocks of agreement.

In [4]:
# Compute Spearman correlation matrix for all metrics
corr_matrix = df_pc[ALL_METRICS].corr(method="spearman")

# Save full concordance matrix
corr_matrix.to_csv(ANALYSIS_DIR / "cross_perspective_concordance.csv")

# Compute within-perspective vs between-perspective concordance
within_corrs = []
between_corrs = []
for i, m1 in enumerate(ALL_METRICS):
    for j, m2 in enumerate(ALL_METRICS):
        if i >= j:
            continue
        r = corr_matrix.loc[m1, m2]
        if np.isnan(r):
            continue
        p1 = perspective_map.get(m1)
        p2 = perspective_map.get(m2)
        if p1 == p2:
            within_corrs.append(abs(r))
        else:
            between_corrs.append(abs(r))

print(f"Cross-perspective concordance matrix: {corr_matrix.shape}")
print(
    f"\nMean |rho| within perspective:  {np.mean(within_corrs):.3f} (n={len(within_corrs)})"
)
print(
    f"Mean |rho| between perspectives: {np.mean(between_corrs):.3f} (n={len(between_corrs)})"
)
print(f"Ratio within/between: {np.mean(within_corrs) / np.mean(between_corrs):.2f}")

Cross-perspective concordance matrix: (26, 26)

Mean |rho| within perspective:  0.596 (n=101)
Mean |rho| between perspectives: 0.549 (n=224)
Ratio within/between: 1.09


In [5]:
# --- VIZ T4_06: Cross-Perspective Concordance Heatmap ---

# Cluster the correlation matrix
dist_matrix = 1 - corr_matrix.abs().values
np.fill_diagonal(dist_matrix, 0)
# Ensure symmetry and valid distance
dist_matrix = (dist_matrix + dist_matrix.T) / 2
dist_matrix = np.clip(dist_matrix, 0, None)

condensed = squareform(dist_matrix)
Z = linkage(condensed, method="average")
order = leaves_list(Z)

# Reorder
ordered_metrics = [ALL_METRICS[i] for i in order]
ordered_corr = corr_matrix.loc[ordered_metrics, ordered_metrics]

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

im = ax.imshow(ordered_corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
ax.grid(False)

# Labels
ax.set_xticks(range(len(ordered_metrics)))
ax.set_yticks(range(len(ordered_metrics)))

# Color labels by perspective
xlabels = ax.set_xticklabels(ordered_metrics, rotation=60, ha="right", fontsize=8)
ylabels = ax.set_yticklabels(ordered_metrics, fontsize=8)
for label in xlabels:
    metric_name = label.get_text()
    p = perspective_map.get(metric_name, "unknown")
    label.set_color(PERSPECTIVE_COLORS.get(p, "black"))
for label in ylabels:
    metric_name = label.get_text()
    p = perspective_map.get(metric_name, "unknown")
    label.set_color(PERSPECTIVE_COLORS.get(p, "black"))

plt.colorbar(im, ax=ax, label="Spearman rho", shrink=0.8)
ax.set_title(
    "Cross-Perspective Concordance (hierarchically clustered)",
    fontsize=13,
    fontweight="bold",
)

# Legend for perspective colors
import matplotlib.patches as mpatches

legend_patches = [
    mpatches.Patch(color=c, label=p.capitalize()) for p, c in PERSPECTIVE_COLORS.items()
]
ax.legend(
    handles=legend_patches,
    loc="lower left",
    fontsize=9,
    framealpha=0.9,
    title="Perspective",
)

plt.savefig(VIZ_DIR / "T4_06_concordance_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_06_concordance_heatmap.png")

Saved: T4_06_concordance_heatmap.png


## 3. Condition Effect Consistency

For each condition (band), compute its rank on key metrics from each perspective.
Check whether conditions rank consistently across perspectives using Kendall's W.

In [6]:
# Select representative metrics from each perspective
repr_metrics = {
    "circuit_accuracy": "functional",
    "retention_ratio": "functional",
    "edge_fraction": "structural",
    "skip_fraction": "structural",
    "convergence_layer": "representational",
    "final_prob_correct": "representational",
}
repr_metrics = {k: v for k, v in repr_metrics.items() if k in df_pc.columns}


def kendall_w(rankings_matrix):
    """Compute Kendall's W (coefficient of concordance).
    rankings_matrix: (n_raters, n_items) array of ranks.
    """
    k, n = rankings_matrix.shape
    if k < 2 or n < 2:
        return np.nan
    rank_sums = rankings_matrix.sum(axis=0)
    mean_rank_sum = rank_sums.mean()
    ss = np.sum((rank_sums - mean_rank_sum) ** 2)
    w = 12 * ss / (k**2 * (n**3 - n))
    return w


cond_rows = []
kendall_rows = []

for model in MODELS:
    df_m = df_pc[df_pc["model"] == model].copy()
    # Mean across replications for each condition
    df_cond = df_m.groupby("band")[list(repr_metrics.keys())].mean()

    # Rank conditions for each metric (1 = best)
    rankings = pd.DataFrame(index=df_cond.index)
    for metric in repr_metrics:
        # For most metrics, higher is better; for convergence_layer and edge_fraction, interpret contextually
        # Use rank without assuming direction: just compute ordinal rank
        rankings[metric] = df_cond[metric].rank(ascending=False)

    for cond in rankings.index:
        for metric in repr_metrics:
            cond_rows.append(
                {
                    "model": model,
                    "condition": cond,
                    "metric": metric,
                    "perspective": repr_metrics[metric],
                    "value": df_cond.loc[cond, metric],
                    "rank": rankings.loc[cond, metric],
                }
            )

    # Kendall's W across metrics for this model
    rank_matrix = rankings.values.T  # (n_metrics, n_conditions)
    w = kendall_w(rank_matrix)
    kendall_rows.append(
        {
            "model": model,
            "kendall_w": w,
            "n_metrics": len(repr_metrics),
            "n_conditions": len(CONDITIONS),
        }
    )

df_cond_ranking = pd.DataFrame(cond_rows)
df_cond_ranking.to_csv(ANALYSIS_DIR / "condition_ranking_consistency.csv", index=False)

df_kendall = pd.DataFrame(kendall_rows)
print("Kendall's W (condition ranking concordance across perspectives):")
print(df_kendall.to_string(index=False))
print(f"\nMean Kendall's W: {df_kendall['kendall_w'].mean():.3f}")
print("  (W=1.0 means perfect agreement, W=0 means no agreement)")

Kendall's W (condition ranking concordance across perspectives):
      model  kendall_w  n_metrics  n_conditions
pythia-1.4b   0.077778          6             4
pythia-160m   0.033333          6             4
  pythia-1b   0.211111          6             4
pythia-410m   0.500000          6             4
 pythia-70m   0.188889          6             4

Mean Kendall's W: 0.202
  (W=1.0 means perfect agreement, W=0 means no agreement)


In [7]:
# --- VIZ T4_07: Condition Ranking Consistency ---

fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

metric_names = list(repr_metrics.keys())

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    df_m = df_cond_ranking[df_cond_ranking["model"] == model]

    # Pivot: conditions x metrics with rank as value
    pivot = df_m.pivot(index="condition", columns="metric", values="rank")
    pivot = pivot[metric_names]  # consistent order

    # Rename columns for display (replace underscores with newlines)
    pivot_display = pivot.copy()
    pivot_display.columns = [m.replace("_", "\n") for m in pivot.columns]

    sns.heatmap(
        pivot_display,
        ax=ax,
        cmap="RdYlGn_r",
        vmin=1,
        vmax=len(CONDITIONS),
        annot=True,
        fmt=".0f",
        square=True,
        linewidths=0,
        linecolor="none",
        cbar=False,
    )

    # Adjust tick label sizes
    ax.tick_params(axis="x", labelsize=7, rotation=0)
    ax.tick_params(axis="y", labelsize=9, rotation=0)

    w = df_kendall[df_kendall["model"] == model]["kendall_w"].values[0]
    ax.set_title(f"{model}\nW={w:.2f}", fontsize=10, fontweight="bold")

fig.suptitle(
    "Condition Ranking Consistency Across Perspectives",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig(VIZ_DIR / "T4_07_condition_ranking.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_07_condition_ranking.png")

Saved: T4_07_condition_ranking.png


## 4. Scaling Consistency

Do functional, structural, and representational metrics all scale the same way with model size?
For each metric, compute Spearman correlation with log(model_capacity).

In [8]:
# Compute Spearman with log(model_capacity) for each metric
scaling_rows = []
for metric in ALL_METRICS:
    x = np.log10(df_pc["model_capacity"].values)
    y = df_pc[metric].values
    if np.std(y) < 1e-10:
        continue
    rho, p = stats.spearmanr(x, y)
    scaling_rows.append(
        {
            "metric": metric,
            "perspective": perspective_map.get(metric, "unknown"),
            "rho_with_log_capacity": rho,
            "p_value": p,
            "direction": "increases" if rho > 0 else "decreases",
            "significant": p < 0.05,
        }
    )

df_scaling = pd.DataFrame(scaling_rows)
df_scaling.to_csv(ANALYSIS_DIR / "scaling_consistency.csv", index=False)

print("Scaling with log(model_capacity):")
print(
    df_scaling[
        [
            "metric",
            "perspective",
            "rho_with_log_capacity",
            "p_value",
            "direction",
            "significant",
        ]
    ].to_string(index=False)
)

print(f"\nScaling direction summary by perspective:")
for p in ["structural", "functional", "representational"]:
    sub = df_scaling[df_scaling["perspective"] == p]
    n_inc = (sub["direction"] == "increases").sum()
    n_dec = (sub["direction"] == "decreases").sum()
    n_sig = sub["significant"].sum()
    print(f"  {p}: {n_inc} increase, {n_dec} decrease, {n_sig}/{len(sub)} significant")

Scaling with log(model_capacity):
                 metric      perspective  rho_with_log_capacity      p_value direction  significant
          edge_fraction       structural              -0.881951 1.331397e-20 decreases         True
          skip_fraction       structural               0.523991 1.733521e-05 increases         True
          attn_fraction       structural               0.698882 5.305914e-10 increases         True
           mlp_fraction       structural              -0.121811 3.538464e-01 decreases        False
         resid_fraction       structural              -0.817302 1.644815e-15 decreases         True
head_participation_rate       structural              -0.875450 5.740165e-20 decreases         True
           active_heads       structural               0.490307 6.987668e-05 increases         True
    mean_edges_per_head       structural               0.388576 2.154066e-03 increases         True
            n_universal       structural               0.196748 1.

In [9]:
# --- VIZ T4_08: Unified Scaling ---

# Select a representative metric from each perspective for visualization
scaling_repr = {
    "retention_ratio": "functional",
    "circuit_accuracy": "functional",
    "edge_fraction": "structural",
    "universal_fraction": "structural",
    "convergence_layer": "representational",
    "peak_probe_accuracy": "representational",
}
scaling_repr = {k: v for k, v in scaling_repr.items() if k in df_pc.columns}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (metric, persp) in enumerate(scaling_repr.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]

    # Model-level mean +/- std
    model_means = df_pc.groupby("model")[metric].agg(["mean", "std"]).reset_index()
    cap_map = {
        "pythia-70m": 70,
        "pythia-160m": 160,
        "pythia-410m": 410,
        "pythia-1b": 1000,
        "pythia-1.4b": 1400,
    }
    model_means["capacity"] = model_means["model"].map(cap_map)
    model_means = model_means.sort_values("capacity")

    ax.errorbar(
        np.log10(model_means["capacity"]),
        model_means["mean"],
        yerr=model_means["std"],
        fmt="o-",
        color=PERSPECTIVE_COLORS[persp],
        capsize=4,
        markersize=8,
        lw=2,
    )

    # Also scatter individual points
    for model in MODELS:
        mask = df_pc["model"] == model
        cap = np.log10(cap_map[model])
        jitter = np.random.uniform(-0.02, 0.02, mask.sum())
        ax.scatter(
            cap + jitter,
            df_pc.loc[mask, metric],
            c=PERSPECTIVE_COLORS[persp],
            alpha=0.3,
            s=20,
            zorder=1,
        )

    # Annotate with rho
    rho_row = df_scaling[df_scaling["metric"] == metric]
    if len(rho_row) > 0:
        rho = rho_row.iloc[0]["rho_with_log_capacity"]
        p_val = rho_row.iloc[0]["p_value"]
        sig_str = (
            "***"
            if p_val < 0.001
            else "**"
            if p_val < 0.01
            else "*"
            if p_val < 0.05
            else "ns"
        )
        ax.text(
            0.05,
            0.95,
            f"rho={rho:.2f} {sig_str}",
            transform=ax.transAxes,
            fontsize=9,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    ax.set_xlabel("log10(params, M)", fontsize=10)
    ax.set_ylabel(metric.replace("_", " "), fontsize=10)
    ax.set_title(f"{metric} ({persp[0].upper()})", fontsize=11, fontweight="bold")
    ax.set_xticks(np.log10([70, 160, 410, 1000]))
    ax.set_xticklabels(["70M", "160M", "410M", "1B"], fontsize=9)

# Hide unused axes
for idx in range(len(scaling_repr), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Scaling Consistency Across Perspectives", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(VIZ_DIR / "T4_08_scaling_consistency.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: T4_08_scaling_consistency.png")

Saved: T4_08_scaling_consistency.png


## 5. Perspective Disagreement Detection

Identify circuits where perspectives disagree most: high on one perspective's metrics
but low on another's. Computed as sum of pairwise rank differences across perspectives.

In [10]:
# For each circuit, compute a representative rank for each perspective
# Use mean percentile rank across metrics within each perspective

s_cols = [m for m in S_METRICS if m in df_pc.columns and df_pc[m].std() > 1e-10]
f_cols = [m for m in F_METRICS if m in df_pc.columns and df_pc[m].std() > 1e-10]
r_cols = [m for m in R_METRICS if m in df_pc.columns and df_pc[m].std() > 1e-10]

# Percentile rank each metric (0-1, higher = more of that metric)
df_ranks = df_pc[["model", "band", "draw"]].copy()

for col in s_cols:
    df_ranks[f"{col}_pctrank"] = df_pc[col].rank(pct=True)
for col in f_cols:
    df_ranks[f"{col}_pctrank"] = df_pc[col].rank(pct=True)
for col in r_cols:
    df_ranks[f"{col}_pctrank"] = df_pc[col].rank(pct=True)

# Mean percentile rank per perspective
df_ranks["rank_S"] = df_ranks[[f"{c}_pctrank" for c in s_cols]].mean(axis=1)
df_ranks["rank_F"] = df_ranks[[f"{c}_pctrank" for c in f_cols]].mean(axis=1)
df_ranks["rank_R"] = df_ranks[[f"{c}_pctrank" for c in r_cols]].mean(axis=1)

# Disagreement score: sum of pairwise absolute rank differences
df_ranks["disagreement"] = (
    (df_ranks["rank_S"] - df_ranks["rank_F"]).abs()
    + (df_ranks["rank_S"] - df_ranks["rank_R"]).abs()
    + (df_ranks["rank_F"] - df_ranks["rank_R"]).abs()
)

df_disagree = df_ranks[
    ["model", "band", "draw", "rank_S", "rank_F", "rank_R", "disagreement"]
].copy()
df_disagree = df_disagree.sort_values("disagreement", ascending=False).reset_index(
    drop=True
)
df_disagree.to_csv(ANALYSIS_DIR / "perspective_disagreements.csv", index=False)

print('Top 10 most "paradoxical" circuits (highest perspective disagreement):')
print(df_disagree.head(10).to_string(index=False))

print(f"\nDisagreement stats:")
print(f"  Mean disagreement: {df_disagree['disagreement'].mean():.3f}")
print(f"  Std disagreement:  {df_disagree['disagreement'].std():.3f}")
print(f"  Max disagreement:  {df_disagree['disagreement'].max():.3f}")
print(f"  Min disagreement:  {df_disagree['disagreement'].min():.3f}")

Top 10 most "paradoxical" circuits (highest perspective disagreement):
      model      band   draw   rank_S   rank_F   rank_R  disagreement
pythia-1.4b       low draw_2 0.435000 0.514583 0.800000      0.730000
pythia-1.4b very_high draw_2 0.385000 0.572917 0.743750      0.717500
pythia-1.4b    medium draw_2 0.426667 0.506250 0.766667      0.680000
pythia-1.4b      high draw_2 0.411667 0.503125 0.737500      0.651667
pythia-160m    medium draw_2 0.666667 0.547917 0.343750      0.645833
pythia-1.4b    medium draw_3 0.433333 0.512500 0.739583      0.612500
pythia-160m    medium draw_3 0.658333 0.471875 0.356250      0.604167
pythia-1.4b very_high draw_3 0.396667 0.615625 0.691667      0.590000
pythia-1.4b      high draw_3 0.418333 0.576042 0.710417      0.584167
 pythia-70m      high draw_2 0.461667 0.295833 0.172917      0.577500

Disagreement stats:
  Mean disagreement: 0.431
  Std disagreement:  0.157
  Max disagreement:  0.730
  Min disagreement:  0.136


## 6. Summary

In [11]:
print("=" * 80)
print("PHASE 4: INTEGRATION 3: CROSS-PERSPECTIVE SYNTHESIS")
print("=" * 80)

print(f"\n--- Metrics ---")
for p in ["structural", "functional", "representational"]:
    n = sum(1 for m in ALL_METRICS if perspective_map.get(m) == p)
    print(f"  {p}: {n} metrics")

print(f"\n--- Variance Decomposition ---")
for p in ["structural", "functional", "representational"]:
    sub = df_var[df_var["perspective"] == p]
    print(
        f"  {p}: model={sub['eta2_model'].mean():.1%}, "
        f"condition={sub['eta2_condition'].mean():.1%}, "
        f"replication={sub['eta2_replication'].mean():.1%}"
    )
dom = df_var["dominant_factor"].value_counts()
print(f"  Dominant factor: {dom.to_dict()}")

print(f"\n--- Concordance ---")
print(f"  Mean |rho| within perspective:  {np.mean(within_corrs):.3f}")
print(f"  Mean |rho| between perspectives: {np.mean(between_corrs):.3f}")

print(f"\n--- Condition Ranking Consistency ---")
for _, row in df_kendall.iterrows():
    print(f"  {row['model']}: Kendall's W = {row['kendall_w']:.3f}")
print(f"  Mean W = {df_kendall['kendall_w'].mean():.3f}")

print(f"\n--- Scaling Consistency ---")
n_sig = df_scaling["significant"].sum()
n_pos = (df_scaling[df_scaling["significant"]]["direction"] == "increases").sum()
n_neg = (df_scaling[df_scaling["significant"]]["direction"] == "decreases").sum()
print(f"  {n_sig}/{len(df_scaling)} metrics significantly scale with model size")
print(f"  Of significant: {n_pos} increase, {n_neg} decrease")

print(f"\n--- Perspective Disagreement ---")
print(
    f"  Mean disagreement score: {df_disagree['disagreement'].mean():.3f} (max possible ~ 1.5)"
)
top = df_disagree.iloc[0]
print(
    f"  Most paradoxical: {top['model']} / {top['band']} / {top['draw']} "
    f"(S={top['rank_S']:.2f}, F={top['rank_F']:.2f}, R={top['rank_R']:.2f})"
)

print(f"\n--- Output Files ---")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")
for f in sorted(VIZ_DIR.glob("T4_*")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

print("\nDone.")

PHASE 4: INTEGRATION 3: CROSS-PERSPECTIVE SYNTHESIS

--- Metrics ---
  structural: 10 metrics
  functional: 8 metrics
  representational: 8 metrics

--- Variance Decomposition ---
  structural: model=93.7%, condition=1.1%, replication=0.2%
  functional: model=80.6%, condition=2.3%, replication=0.9%
  representational: model=95.9%, condition=0.4%, replication=0.3%
  Dominant factor: {'model': 25, 'replication': 1}

--- Concordance ---
  Mean |rho| within perspective:  0.596
  Mean |rho| between perspectives: 0.549

--- Condition Ranking Consistency ---
  pythia-1.4b: Kendall's W = 0.078
  pythia-160m: Kendall's W = 0.033
  pythia-1b: Kendall's W = 0.211
  pythia-410m: Kendall's W = 0.500
  pythia-70m: Kendall's W = 0.189
  Mean W = 0.202

--- Scaling Consistency ---
  22/26 metrics significantly scale with model size
  Of significant: 16 increase, 6 decrease

--- Perspective Disagreement ---
  Mean disagreement score: 0.431 (max possible ~ 1.5)
  Most paradoxical: pythia-1.4b / low / dr